In [1]:
import re, json, time, requests
import pandas as pd

from fastai.vision.all import *
from pathlib import Path
from fastdownload import download_url

In [6]:
HEADERS = {"User-Agent": "personal-learning-project"}
CACHE = Path("cache")  # Saved API response
COVERS = Path("covers")  # covers/{era}/{genre}/xxx.jpg
PROBE = Path("probe")  # Classics
# In ML, a "probe set" means examples you run through a model to poke at its behaviour, as opposed to training or validating it. 

In [ ]:
ERAS = {
    "2000_2004": (2000, 2004),
    "2010_2014": (2010, 2014),
    "2021_2025": (2021, 2025),
}

In [4]:
GENRES = {
    "romance":  ["romance", "love stories"],
    "fantasy":  ["fantasy", "magic"],
    "thriller": ["thriller", "suspense", "mystery", "crime"],
    "scifi":    ["science fiction"],
    "horror":   ["horror"],
}

In [ ]:
N_PER_GENRE_PER_ERA = 100     # same in every era to keep it fair
MAX_PER_AUTHOR = 2  # To avoid the model learning by author name

In [2]:
def search_books(genre, start_year, end_year, page=1):
    """search.json with q=f'subject:{genre} AND first_publish_year:[{start} TO {end}]'.
    Return the list of docs (key, title, author_name, first_publish_year, subject)."""
    # we search by "science fiction" rather than "sci-fi"
    keywords = " OR ".join(f'"{g}"' for g in GENRES[genre])  # we need the quotes as subject: science fiction and 
    # subject:"science fiction" return different results with the former also matching science as a subject and 
    # matching fiction against any field
    params = {
        "q": f"subject: ({keywords}) AND first_publish_year: [{start_year} TO {end_year}]",
        "fields": "key,title,author_name,first_publish_year,subject",
        "limit": 100,
        "page": page,
    }

    # https://openlibrary.org/search.json?q=subject%3a%22romance%22+AND+first_publish_year%3a%5b2021+TO+2025%5d&limit=100&page=1
    
    # fields controls what comes back in the response - we choose the columns we want otherwise we get a lot of unnecessary
    # fields which makes the response bigger and slower
    
    # Lets say there are 50,000 matching books, the server won't send them all at once, so it splits them into pages
    # where there are 100 books per page (limit=100) and page=1 gives books 1-100, page=2 gives books 101-200 and so on
    # This is why the main loop does page += 1 until we have enough books
    r = requests.get("https://openlibrary.org/search.json", 
                    params=params, headers=HEADERS, timeout=30)

    time.sleep(1)
    # Open Library is a free service run by a non-profit (the Internet Archive). Firing hundreds of requests per second can
    # overload it, and it may block you temporarily (HTTP 429 "Too Many Requests"). Waiting one second between requests keeps you safely under that.
    
    return r.json()["docs"]

In [7]:
# CHECK
docs = search_books("romance", 2021, 2025)
len(docs), docs[0]

(100,
 {'author_name': ['Joana Marcus'],
  'first_publish_year': 2021,
  'key': '/works/OL26128903W',
  'title': 'Before December',
  'subject': ['Romance',
   'Christmas romance',
   'second chance romance',
   'forbidden love',
   'family',
   'friendship',
   'love triangle',
   'secrets',
   'lies',
   'forgiveness',
   'redemption',
   'love story',
   'holiday romance',
   'small town romance']})

In [22]:
def subjects_to_genres(subjects):
    """['Fiction, romance, contemporary', 'Magic'] -> {'romance', 'fantasy'}"""
    found = set()
    for subject in subjects:
        for genre, keywords in GENRES.items():
            for keyword in keywords:              # check each keyword separately
                if keyword in subject.lower():
                    found.add(genre)

    return found

In [23]:
# CHECK — should print {'romance', 'fantasy'}
subjects_to_genres(["Fiction, romance, contemporary", "Magic"])

{'fantasy', 'romance'}

In [ ]:
books = []
seen = set()  # (title, author) pairs we already have
author_count = {}  # author -> how many of their books we have

for era, (start_year, end_year) in ERAS.items():
    for genre in GENRES:
        count = 0
        page = 1
        while count < N_PER_GENRE_PER_ERA:
            # - page through search_works until you have N_PER_GENRE_PER_ERA books
            docs = search_books(genre, start_year, end_year, page)

            if len(docs) == 0:  # no more results
                break

            for doc in docs:
                title  = doc.get("title", "")
                author = doc.get("author_name", ["unknown"])[0]
                genres = subjects_to_genres(doc.get("subject", []))
                book_id = (title.lower(), author.lower())

            # TODO: if book_id is in seen -> continue
            # TODO: if author_count.get(author, 0) >= MAX_PER_AUTHOR -> continue
            # TODO: if genres is empty -> continue

            books.append(dict(
                work_key   = doc["key"],
                title      = title,
                author     = author,
                year       = doc["first_publish_year"],
                era        = era,
                searched   = genre,
                all_genres = " ".join(sorted(genres)),
                n_genres   = len(genres),
            ))

            # TODO: add book_id to seen
            # TODO: add 1 to author_count for this author
            count += 1
            page += 1